# ML 工程：一次准备，保存信号，多次账户消费

从固定真实样本出发，这一页沿输入、Feature、成熟标签、模型、保存 Signal、账户和展示走完一条短流程。固定配置首建/HIT、两周控制试点与同 Signal Top3/Top5 对照已通过；全年和五年尚未运行。收益和参数优劣不作为工程通过条件。

正文合同归 [Research](../docs/design/05_axiom_research.md)、[Core](../docs/design/03_axiom_core.md)、[Trade](../docs/design/04_axiom_trade.md)。本教程展示如何消费这些合同。

## 1. 数据与计算走哪条路

![Axiom ML 架构](../docs/design/ml-engineering-assets/architecture.svg)

实线表示已有有界实现；虚线表示通用 ModelHandle/SignalPlan、公开滚动与有界缓存目标。模型训练与推理目前由 Research 编排，Core 提供共享 Feature/组合计算，Runtime 唯一处理账户。


## 2. 固定真实短样本：先知道正在读什么

January 基线有 314 个固定预测 ID、六特征和真实交易日；账户执行范围仍为已准入的 SZSE 主板子集。完整来源、历史成员、cutoff 与缺失限制随 owner 保存件保留。这不是以后各月完整 CSI300。

下面读取本轮保存的教学回执，展示规模、成熟边界和实际次数。它不重新训练或读取 Data。独立执行的完整输入与身份留在调用者本地；公开 source 只接收环境变量，不包含机器路径或私人结果。

In [ ]:
import hashlib
import json
import os
from pathlib import Path

receipt_path = Path(os.environ["AXIOM_ML_TEACHING_RECEIPT"])
expected = os.environ["AXIOM_ML_TEACHING_RECEIPT_REF"]
raw = receipt_path.read_bytes()
assert "sha256:" + hashlib.sha256(raw).hexdigest() == expected
receipt = json.loads(raw)
assert receipt["contract_version"] == "private_ml_teaching_acceptance_v1"
print("样本：固定 January 314 ID、六特征；weekly 使用固定 11 月至 1 月日期")
print("当前读取保存回执，不重跑 Data、Feature、模型或账户。")


## 3. Feature ID 与依赖：表从 registry 自动生成

下面表来自已合 Research 的 `stock_ml_v1.json`，不另维护手写公式源。ID 按语义分组，不要求连续；列顺序由本模型实际 feature_selection 决定。表的选择与版本已按保存模型和公共 catalog 核对；编译器校验 lookback 与 Core 依赖闭包。

| 分组 | ID / 名称 | 公式或含义 | 依赖字段 | Lookback / session | 语义版本 | 本模型 |
|---|---|---|---|---|---|---|
| MOM | MOM010 / ret1 | adjusted_close[t] / adjusted_close[t-1] - 1; common anchor = feature session | market_daily.close, adjustment_factors.factor | 2 | 1.0.0 | 选用 |
| MOM | MOM020 / ret5 | adjusted_close[t] / adjusted_close[t-5] - 1; common anchor = feature session | market_daily.close, adjustment_factors.factor | 6 | 1.0.0 | 选用 |
| MOM | MOM030 / ret20 | adjusted_close[t] / adjusted_close[t-20] - 1; common anchor = feature session | market_daily.close, adjustment_factors.factor | 21 | 1.0.0 | 选用 |
| VOL | VOL010 / intraday_range | (high[t] - low[t]) / close[t] | market_daily.high, market_daily.low, market_daily.close, adjustment_factors.factor | 1 | 1.0.0 | 选用 |
| PRC | PRC010 / close_to_open | close[t] / open[t] - 1 | market_daily.close, market_daily.open, adjustment_factors.factor | 1 | 1.0.0 | 选用 |
| LIQ | LIQ010 / amount_relative_5 | amount_cny[t] / rolling_mean(amount_cny, window=5, inclusive_current=true) | market_daily.amount_cny | 5 | 1.0.0 | 选用 |

[固定 catalog 源](https://github.com/sinnergarden/axiom-research/blob/90b378a6e7d8283b0b021c537a95c5468b9161bc/src/axiom_research/catalogs/stock_ml_v1.json)。

## 4. Data / Qlib / FeatureBuild：准备一次，校验后复用

Data 将 Raw 规范为固定 Snapshot。QuerySpec 冻结字段、成员、日历、PIT 与逐 session cutoff；Qlib 导出生成格式数值及完整来源证明，尚未计算预测 Feature。Research catalog 编译既有 Core FeaturePlan，由共享执行器生成 FeatureBuild。

本次使用完整原 Feature parent 与证明，无供应商调用，也没有重算 Feature。保存件加载仍有解析/hash/ref 成本：首建从保存输入生成模型，第二次相同目标的 HIT 也需校验文件。冷 Data/Qlib 准备引用历史实测，不能冒称本次冷跑。

## 5. 五 session label：成熟之后才能进入训练

标签是实际日历的 f+1 开盘到 f+5 收盘收益。每个 fit clock 只接纳已成熟、当时成员且六特征有效的样本；归一化使用同一天完整有效截面的 cs_zscore（ddof=0、无 clip）。模型预测是该标准化目标的无量纲分数。

January 原 fit 为 2023-12-28 20:30 +08，最后成熟 Feature 日为 12-21，训练 16,200 行。一个已保存日期的算术例：原收益 −0.01608579，截面均值 −0.01783237、标准差 0.03731907，标准化值约 0.04680116。这里解释原 Core 输出，没有另算 Feature 或归一化执行器。

两周控制试点的训练日期同为 2023-11-01—2024-01-31，但 02-02 fit 最后成熟日是 01-26，02-08 fit 可到 01-31。训练时钟之外的 outcome 不进入 Dataset。

## 6. Dataset 与模型：实际首建，然后实际 HIT

`build_stock_ml_from_saved_features` 复用固定同配置 Feature/raw Label，第一次新目标重新生成 norm/Dataset 并 fit/infer；第二次完全相同目标通过真实 builder 命中。它不是跨配置滚动 builder。

| 实际阶段 | Wall time | fit / infer | Data / FeatureCore |
|---|---:|---:|---:|
| 新目标首建 | 13.14 s | 1 / 1 | 0 / 0 |
| 同目标 HIT | 4.95 s | 0 / 0 | 0 / 0 |

首建内部 norm/Dataset 约 5.61 s，fit 68.50 ms、infer 12.36 ms；内部阶段嵌套，不能再与总时间相加。两次完整过程 18.43 s，峰值约 1.17 GiB，新产物约 279 MiB。HIT 的 4.95 s 说明保存件校验仍有 I/O。

以下函数对应已执行的公开调用；本 Notebook 默认只读取已保存回执，避免为了展示再次训练。另开新受控试验时才调用此函数。

In [ ]:
def first_build_then_hit(experiment_path, new_destination):
    from axiom_research import build_stock_ml_from_saved_features
    first_metrics, hit_metrics = {}, {}
    first = build_stock_ml_from_saved_features(
        experiment_path, destination=new_destination, metrics=first_metrics)
    second = build_stock_ml_from_saved_features(
        experiment_path, destination=new_destination, metrics=hit_metrics)
    assert not first_metrics["cache_hit"] and hit_metrics["cache_hit"]
    assert first.identity == second.identity
    assert first_metrics["train_calls"] == first_metrics["predict_calls"] == 1
    assert all(hit_metrics[k] == 0 for k in (
        "train_calls", "predict_calls", "data_read_calls", "feature_core_calls"))
    return first_metrics, hit_metrics

for row in receipt["fixed_config"]["stages"]:
    print(row["stage"], round(row["elapsed_seconds"], 3), "s", row["metrics"])


## 7. 保存 Signal：分数与账户目标分开

公开 January Signal 保留 model/Feature 引用、完整键、成员、validity 与模拟可用时点，不保存账户持仓路径或撮合结果。Engine 消费严格前一实际 session 的预测，不能回填未来模型分数。

纯预测 Signal 可供独立账户复用。新组合生成新的计划和账本；账户路径、退出与风控状态各属自己的账户。两周试点目前是私有 native booster/预测回执，尚非通用公开 ModelRelease/SignalRun，不将它交给账户冒充完整滚动策略。

## 8. Engine 唯一账户 / Evaluation：同 Signal 换组合

**同一保存 Signal 的 Top3/Top5 对照已通过。** 两份新账户与原 v2 评价均已保存，五个调仓日目标均不同。Data、供应商、Feature、fit、predict 调用全部为 0；完整输入与评价引用在本地回执逐项核对。

| 实际调仓日 | Top5 选中数 | Top3 选中数 | 目标组合核对 |
|---|---:|---:|---|
| 2024-01-02 | 5 | 3 | 不同 |
| 2024-01-08 | 5 | 3 | 不同 |
| 2024-01-15 | 5 | 3 | 不同 |
| 2024-01-22 | 5 | 3 | 不同 |
| 2024-01-29 | 5 | 3 | 不同 |

| 固定不变 | 实际变化 | 保存结果 |
|---|---|---|
| 原 January Signal / MarketReplay / 初始资金 / profile / 费用 / T+1 / lot / 执行范围与来源限制 | 仅 portfolio_policy.top_k：5 → 3 | 独立 run_id、完整账户与评价；原 Signal 和旧结果不重写 |
| Data / 供应商 / Feature / fit / predict | 均为 0 次 | 上游未导入、未调用；实际 Runtime / Evaluation 输出保留 |

TopK 合同已由 Docs PR17 审准，Engine PR8 已合。实际绑定已审源码 `f6d93a1`；`stock_portfolio_policy(top_k=k, execution_universe=原执行集合)` 返回现有请求政策，替换此字段后仍沿 `BacktestRequest.from_dict` / `run_backtest`。所有显式 k 使用 Core /2，旧 /1 Top5 保存件仍可读取。准确执行与统计定义见 [Trade](../docs/design/04_axiom_trade.md#61-有界股票日线-top5)。本例沿用固定 v2 Evaluation 输入，未混入后来新增的评价版本。

首次进程同时保留旧账户/评价、新账户及其展开字典，评价触发 RSS guard，失败记录保留。恢复将账户与评价分为独立进程：先核验已保存 Top5，再仅补 Top5 评价、Top3 账户和 Top3 评价，**没有重复 Top5 账户**。每阶段释放不用的对象并保留完整 proof，恢复 106.65 s、峰值 4.78 GiB，全部进程已退出。这处理本样本的生命周期峰值，尚不能证明多年 loader 内存有界。

下面两个函数对应已验收的阶段调用；必须在绑定已审 Engine 来源的受控新进程中分别调用。默认只定义函数并读取回执，已完成阶段不重复执行。


In [ ]:
def make_topk_account(saved_run_path, top_k, new_destination):
    import gc
    from axiom_engine.runtime import (load_backtest_run, stock_portfolio_policy,
        BacktestRequest, run_backtest, save_backtest_run)

    output = Path(new_destination)
    output.mkdir(parents=True, exist_ok=False)
    baseline = load_backtest_run(saved_run_path)
    plan = baseline.to_dict()["plan"]
    del baseline
    gc.collect()
    plan["portfolio_policy"] = stock_portfolio_policy(
        top_k=top_k, execution_universe=plan["execution_universe"])
    request = BacktestRequest.from_dict(plan)
    del plan
    gc.collect()
    run = run_backtest(request)
    del request
    gc.collect()
    save_backtest_run(run, output / "backtest.json")
    return output / "backtest.json"


def evaluate_saved_topk(saved_run_path, saved_reference_evaluation, new_destination):
    import gc
    from axiom_engine.runtime import (load_backtest_run, load_backtest_evaluation,
        BenchmarkSeries, DividendScope, EvaluationSpec, evaluate_backtest,
        save_backtest_evaluation)

    output = Path(new_destination)
    output.mkdir(parents=True, exist_ok=False)
    report = load_backtest_evaluation(saved_reference_evaluation).to_dict()
    benchmark = BenchmarkSeries.from_dict(report["benchmark_input"])
    scope = None if report["dividend_scope"] is None else DividendScope.from_dict(report["dividend_scope"])
    spec = EvaluationSpec.from_dict(report["spec"])
    del report
    gc.collect()
    run = load_backtest_run(saved_run_path)
    evaluation = evaluate_backtest(run, benchmark=benchmark, spec=spec, dividend_scope=scope)
    del run, benchmark, scope, spec
    gc.collect()
    save_backtest_evaluation(evaluation, output / "evaluation.json")
    return output / "evaluation.json"


# 账户和评价各用一个新进程；本页不调用上述业务函数。
print("TopK 对照状态：", receipt["topk"]["status"])
for result in receipt["topk"]["outputs"]:
    print("Top", result["top_k"], "决策", result["decision_count"], "成交", result["fills"])
print("目标不同的调仓日：", ", ".join(receipt["topk"]["different_target_sessions"]))
print("上游实际调用：", receipt["topk"]["upstream_calls"])
print("账户调用 / 评价尝试 / 完整评价：",
      receipt["cumulative_counts"]["account"],
      receipt["cumulative_counts"]["evaluation_attempt"],
      receipt["cumulative_counts"]["completed_evaluation"])


## 9. UI：保存值进入同一个网页入口

UI Reader 校验 owner 保存件后生成投影；已导出 HTML 打开和切页不执行 Data、模型或账户。Pages 入口由既有发布流程维护，本教程不部署 UI。

源码审阅、owner 浏览器 QA 与用户产品验收是不同证据。父任务已查看实际 UI 视觉范围，用户明确未通过产品验收，整改仍在进行；不能把它登记成“尚未验收”或用模型试点通过替代。

## 10. 重跑边界与已有测量：每种成本分开看

| 改动 / 动作 | 应复用 | 实际需要执行 / 当前限制 |
|---|---|---|
| 只打开已导出 HTML | 所有 owner 结果 | 仅浏览器；不跑 Reader / train / account |
| 加载保存模型 / Signal / run | 原结果内容 | 解析、hash/ref/证明核验；不是零成本 |
| 完全相同 builder 目标 | 全实验 | 公共调用命中后 Data / Core / fit / predict 为 0；仍完整加载核验 |
| 同一独立预测 Signal 改组合 | Data / Feature / Model / Signal | Core 组合 / Runtime / Evaluation；同 Signal Top3/Top5 本轮通过，上游 0 次 |
| 改 label / 模型语义 | 未受影响 Feature / 原事实 | 新 Dataset / model / Signal；现公开保存输入 helper 限同 config |
| 改 Feature 公式 / 字段、Snapshot、PIT/cutoff | 仅实际未受影响阶段 | 新语义链；新增字段或导出 query 变化需新 Qlib view |
| 非语义 Python 修改 | 旧保存件仍可读 | wholeimplementation 可能过粗 MISS；不能报告为语义缓存能力 |

| 既有真实测量，未在本稿重跑 | 观察值 | 可用于解释的范围 |
|---|---|---|
| Data Qlib 314 ID × 56 sessions 冷准备（旧 exporter） | 37.72 s；峰值 5.43 GiB | 独立格式准备；不含模型 / Feature / 账户，新 Data PR39 无生产耗时 / RSS 实测 |
| 有界 monthly 两折 | 282.34 s；峰值 4.09 GiB | February 完整 15 日，March 仅 3/21 日；含额外等价核验，不是 weekly 或全年 |
| 其中 LightGBM fit / infer | fit 61.94 / 42.44 ms；infer 7.79 / 1.58 ms | 单阶段；矩阵/label/来源证明/序列化另算 |
| Engine 一组完整保存账户 + 评价加载 | 27.34 → 14.90 s，下降 45.5% | Reader GROUP 单样本；不是整个网页导出/浏览器速度 |
| 新 Data 导出 manifest 次数 | 冷 5→1；warm 每次重新核验 1 | 算法调用数，不能称 5 倍提速 |

OS 页缓存未清空，单阶段时间有嵌套，不能相加为另一个总时间；没有全年吞吐保证。


## 11. 每周训练控制试点：训练时钟改变，日期保持固定

两个 OOS 窗口为 **2024-02-05—02-08、2024-02-19—02-23**，fit 为严格前一实际 session **02-02/02-08 20:30 +08**。两周训练 Feature 日期均为 **2023-11-01—2024-01-31**，各 65 个实际日期，只重新筛各周已成熟样本；日期窗口相同，成熟样本数不同。这不是每周移动三个月训练窗口的实测。

| fit | 最后成熟 Feature 日 | 成熟日期 / 声明日期 | 训练行 | 有效预测行 | OOS / 预测 Feature 日期数 |
|---|---|---:|---:|---:|---:|
| 2024-02-02 | 2024-01-26 | 62 / 65 | 18,600 | 1,200 | 4 / 4 |
| 2024-02-08 | 2024-01-31 | 65 / 65 | 19,500 | 1,500 | 5 / 5 |

第一周预测 Feature 日期为 02-02/02-05/02-06/02-07，第二周为 02-08/02-19/02-20/02-21/02-22；均对应严格前一 session。两折各 fit/infer 一次，事后标签共同在 **02-29 20:30 +08** 读取，与训练时钟分离。完整日期 parent 保留原 refs，不伪造一个混合的公开 FeatureBuild；这些私有教学预测也没有冒充原 January 公共 SignalRun。

成功流程 **30.94 s、峰值 RSS 4.46 GiB**；矩阵分别约 **1.05/1.11 MiB**。保存 proof 校验阶段采样峰值约 0.68 GiB，Data outcome 与 norm 阶段约 4.46 GiB，fit 阶段 RSS 仍保留之前分配；矩阵小不能解释为进程内存小。本次不是流式 loader 验收。


In [ ]:
# 教学编排调用已有公开 Data/Research/Core，不是另一个特征执行器。
# 仅新的受控试验调用；默认读取本轮已保存统计，不重复 fit/infer。
def two_week_control(frozen_input_manifest, new_destination):
    from examples.weekly_ml_pilot import run_weekly_pilot
    return run_weekly_pilot(frozen_input_manifest, new_destination)

for fold in receipt["weekly"]["folds"]:
    print(fold["fit_cutoff"], "训练", fold["training_rows"],
          "最后成熟日", fold["last_mature_feature_session"],
          "预测", fold["prediction_rows"])
print("本轮含失败尝试的 Data 读取：", receipt["cumulative_counts"]["Data_reads"])


## 12. 五年计划：固定规模条件估算与未测成本

**65.15–66.87 分钟只代表暖流程的固定日期控制情景，不能当五年滑动训练总耗时。** 输入条件是同硬件、314 个固定 ID、六特征、已准备 Feature、固定 65 个训练日期，以及当前短样本的本地源/proof 体积。五年真实移动窗口会改变日期、成员、证明和 I/O；本轮没有启动全年或五年任务。

从已有保存日历枚举 **2021-01-01—2025-12-31**，实际首日为 2021-01-04，末日为 2025-12-31：共有 **1,212 sessions / 256 个有交易日的 ISO 周**，全周休市不算 fold。平均每周 **4.734375** 日，与两周控制平均 4.5 日不同。此日历只用于计数，不证明五年股票源或历史成员覆盖。

以下均从同一成功控制流程的保存分阶段回执拆分，单位为秒；没有额外运行。`V=2.87385` 为公开实验、scope 和三个 parent 的完整核验之和；共享事后 labels 为 `9.98296`、norm 为 `0.90769`；两周训练 labels/norm/matrix/fit/infer 各自总和为 `7.56586 / 7.96881`。

| 暖流程分项 | 条件外推公式（分钟） | 256 周参考值 |
|---|---|---:|
| 每周训练准备 / norm / matrix / fit / infer | `256 × mean(7.56586, 7.96881) / 60` | 33.14 min |
| 共享事后 labels / norm | `256 × (9.98296 + 0.90769) / 2 / 60` | 23.23 min |
| 原 parent/proof 核验 | `256 × V / 2 / 60` | 6.13 min |
| import、检查、序列化与监控剩余时间 | `66.01 − 上述三项` | 3.50 min |
| 整个短流程 | `256 / 2 × 30.94023 / 60` | **66.01 min** |

fit/infer 已包含在训练分项内；仅这两个阶段的外推合计约 **16.63 秒**，不能再加一次，也不能替代总流程。训练分项取两个观测的最小/最大值、其余三项固定，得到 **[65.15, 66.87] min**。这是两周控制情景区间，不是统计置信界、真实五年滑动运行时或承诺上界。

完整计划必须写为：

`T_total = T_唯一冷准备 + T_暖控制[65.15,66.87] min + T_五年账户及评价 + T_导出展示`

| 总计划成本 | 本轮是否已测 | 估计限制 |
|---|---|---|
| 唯一日期的 Data / Qlib / Feature 冷准备 | **未测五年规模** | 旧 314 ID × 56 日 Qlib 的 37.72 s 只说明短样本格式准备，不含全部冷链 |
| 暖 weekly 控制 | 两周实测，256 周条件外推 | 固定日期与体积；不证明移动训练、五年成员、source proof 增长或流式 loader |
| 五年账户及评价 | **未测** | January TopK 仅证明两策略消费同 Signal；恢复 106.65 s 不能线性承诺五年 Runtime |
| 导出展示 | **未测完整流程** | Reader 小样本加载时间不等于网页导出与浏览器耗时 |

因此总计划区间为 **[65.15,66.87] min + 三个未测项**，目前没有可证的有限总上界。模型算术很小，成熟筛选/norm 与源/proof I/O 已占主要成本；历史 union 与 source proof 扩张会增加校验及驻留内存。正式按日期分片加载、阶段缓存与恢复合同仍待实现，不能用短样本准备后速度掩盖冷准备或增长成本。


## 13. 工程附录：预算、失败和候选合同

初始授权为同配置 1 次 fit/infer + 第二次 HIT、两周各 1 次 fit/infer、公共 label outcome 查询最多 8 次、供应商 0；账户最多 2 次。一个重进程，硬 RSS 6 GiB、自有 PID 在 5.5 GiB 停止；每阶段 120 s、业务时钟 480 s、累计新产物 1 GiB。

两次 weekly 失败均发生在查询证据序列化，各只读 1 个价格 batch、未训练。修复后两周共享事后 outcome 来源，成功流程 6 次读取；累计 8，失败目录保留。业务并非一次无错完成，缓存首建和 weekly 也分别计时。

TopK 首次评价采样峰值 **5.82 GiB**，触发 own-PID guard 后退出，Top5 账户已完整保存、Top3 尚未开始；该失败记录及产物没有覆盖。审准的恢复仅补缺少阶段，四个独立进程共用额外 480 s，不重跑 Top5，仍沿用相同 RSS/累计文件上限。实际恢复 **106.65 s / 4.78 GiB**，累计文件约 **674 MiB**，全部进程退出。

| 累计次数（含失败与恢复） | 实际 |
|---|---:|
| Data 读取 / 供应商请求 | 8 / 0 |
| Feature Core 执行 | 0 |
| 固定配置 + weekly fit / predict | 3 / 3 |
| TopK 独立账户 | 2 |
| TopK 评价尝试 / 完整评价 | 3 / 2 |

仅 TopK 的 Data/供应商/Feature/fit/predict 均为 0；label/norm 使用既有 Core 算子，不能把“Feature Core 0”解释为所有 Core 计算 0。Notebook 当前四个默认代码单元只读取保存回执、定义函数，零业务调用；私有输出副本与全部原输入保留，本公开 source 不包含私人产物。

缓存 key 与 proof 有界加载的最小候选见 [Research §8.4.1](../docs/design/05_axiom_research.md#ml-engineering-review-proposals)：实际计划决定上下文；两日期 shard 只为本次配置；顶层调用重验、调用内 proof blob 验一次后共享并释放。落盘本身不证明 streaming，正式 rolling/resume 尚未实现。冷 Qlib 准备沿用历史证据，未重跑。


### 缓存与常驻内存：当前风险

| 现状与风险 | 建议目标 | 小例如何验 |
|---|---|---|
| Research builder 把 Research Python、Data Python 与 Core Python 的整体实现 digest 纳入身份；非语义代码改动也可能 MISS，保存输入再利用仍限同配置 | 后续按实际计算依赖与语义版本拆分身份；保留完整实现来源用于审计。变更前先冻结哪些修改应失效，避免默默宽松复用 | 同输入加载旧保存件仍通过；当前 MISS 如实报告。先列失效矩阵，教程不直接改生产 cache |
| 长历史 Feature / proof 大对象可能一直驻留；两折小例未证明多年内存有界 | 按完整日期截面分片持久化、逐片校验并释放；固定 parent/ref 图与原子 checkpoint | 先完整一周、再代表范围测 RSS / 文件字节 / 中断恢复；未通过不扩全年 |

目标方向已接受，具体最小新 API 仍待冻结；两折不证明完整平台或 streaming 已实现。
四项候选的准确边界统一见 [Research §8.4.1](../docs/design/05_axiom_research.md#ml-engineering-review-proposals)：阶段依赖身份保留实际计算实现与完整审计；首轮按两个完整日期分片、原 proof blob 独立保存并每次核验。它们尚未实施，本教学不先大改缓存或 loader。


### SysQ：借经验，明确已有差异

| 已读证据中的 SysQ 做法 | Axiom 当前实现与取舍 |
|---|---|
| Feature per-window write-through；命中后读原窗口 frame，再训练 | 已有固定单折保存输入复用。本轮跨 fold 的完整日期 parent 编排是有界验收脚本，尚非公开 rolling 缓存平台 |
| SignalStore / LabelStore / 实验索引；信号分析与 signal-driven backtest 分开 | 保存 Signal 后交 Engine 唯一账户；只读 UI 使用 owner 保存件 |
| 已读 research usecase 的成熟门槛与 lineage 检查仍列 TBD | 当前逐行成熟时钟、输入 digest、完整 universe / invalid、分数阶段已明确保留；不沿用工作日 fallback、填零或当前成员替代历史 |

此表沿用 [此前固定 SysQ 对照](../docs/design/05_axiom_research.md#stock-qlib-lightgbm-minimal) 中的 usecase / generator / training 证据，不是新一轮全仓审计或 SysQ 当前全量评价。